# Smart Lender - Loan Eligibility Prediction System

This notebook covers the exploratory data analysis, preprocessing, feature engineering, and model training/evaluation for the Smart Lender system.

### Step 1: Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# Set design system styling
plt.style.use('fivethirtyeight')

### Step 2: Read the Dataset

In [ ]:
data = pd.read_csv('../Dataset/loan_prediction.csv')
print("Dataset Shape:", data.shape)
data.head()

### Step 3: Exploratory Data Analysis (EDA)
Perform univariate, bivariate, and multivariate analysis to understand distribution and correlations.

In [ ]:
# Univariate Analysis - Continuous features distribution
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(data['ApplicantIncome'], color='r', kde=True)
plt.title('Applicant Income Distribution')

plt.subplot(1, 2, 2)
sns.histplot(data['Credit_History'].dropna(), kde=True)
plt.title('Credit History Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Univariate Analysis - Categorical features
plt.figure(figsize=(18, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=data, x='Gender')
plt.title('Gender Distribution')

plt.subplot(1, 2, 2)
sns.countplot(data=data, x='Education')
plt.title('Education Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate Analysis - Segmenting fields
plt.figure(figsize=(20, 5))
plt.subplot(1, 3, 1)
sns.countplot(data=data, x='Married', hue='Gender')
plt.title('Married Segmented by Gender')

plt.subplot(1, 3, 2)
sns.countplot(data=data, x='Self_Employed', hue='Education')
plt.title('Self Employed Segmented by Education')

plt.subplot(1, 3, 3)
sns.countplot(data=data, x='Property_Area', hue='Loan_Amount_Term')
plt.title('Property Area Segmented by Loan Term')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate Analysis - Swarm plot
plt.figure(figsize=(10, 6))
sns.swarmplot(data=data, x='Gender', y='ApplicantIncome', hue='Loan_Status', palette='Set2')
plt.title('Applicant Income & Gender vs Loan Status')
plt.show()

### Step 4: Data Preprocessing & Categorical Encoding

In [ ]:
# Strip '+' from Dependents and fill missing values with mode
data['Dependents'] = data['Dependents'].str.replace('+', '', regex=False)

# Missing value treatment
# Numerical variables -> mean (or mode for discrete/skewed like LoanAmount/Loan_Amount_Term)
data['LoanAmount'] = data['LoanAmount'].fillna(data['LoanAmount'].mean())
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(data['Loan_Amount_Term'].mode()[0])
data['Credit_History'] = data['Credit_History'].fillna(data['Credit_History'].mode()[0])

# Categorical variables -> mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    data[col] = data[col].fillna(data[col].mode()[0])

# Mapping/Encoding categorical columns
gender_map = {'Female': 1, 'Male': 0}
married_map = {'Yes': 1, 'No': 0}
education_map = {'Graduate': 1, 'Not Graduate': 0}
self_employed_map = {'Yes': 1, 'No': 0}
property_area_map = {'Urban': 2, 'Semiurban': 1, 'Rural': 0}
loan_status_map = {'Y': 1, 'N': 0}

data['Gender'] = data['Gender'].map(gender_map)
data['Married'] = data['Married'].map(married_map)
data['Education'] = data['Education'].map(education_map)
data['Self_Employed'] = data['Self_Employed'].map(self_employed_map)
data['Property_Area'] = data['Property_Area'].map(property_area_map)
data['Loan_Status'] = data['Loan_Status'].map(loan_status_map)

# Cast columns to int64 as shown in the Jupyter notebook screenshots
data['Gender'] = data['Gender'].astype('int64')
data['Married'] = data['Married'].astype('int64')
data['Dependents'] = data['Dependents'].astype('int64')
data['Self_Employed'] = data['Self_Employed'].astype('int64')
data['LoanAmount'] = data['LoanAmount'].astype('int64')
data['CoapplicantIncome'] = data['CoapplicantIncome'].astype('int64')
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].astype('int64')
data['Credit_History'] = data['Credit_History'].astype('int64')

print("Remaining missing values:", data.isnull().sum().sum())
data.info()

### Step 5: Class Balancing using SMOTE

In [ ]:
x = data.drop(columns=['Loan_ID', 'Loan_Status'])
y = data['Loan_Status']

print("Target Class counts before SMOTE:")
print(y.value_counts())

smote = SMOTE(random_state=42)
x_bal, y_bal = smote.fit_resample(x, y)

print("\nTarget Class counts after SMOTE:")
print(y_bal.value_counts())

### Step 6: Feature Scaling & Split

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
names = x_bal.columns
x_bal_scaled = scaler.fit_transform(x_bal)

# Save the scaler object to scale1.pkl
with open('../scale1.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved scale1.pkl")

x_bal = pd.DataFrame(x_bal_scaled, columns=names)

# Train-Test Split (test_size = 0.33, random_state = 42)
X_train, X_test, y_train, y_test = train_test_split(x_bal, y_bal, test_size=0.33, random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

### Step 7: Train & Evaluate Classifiers
Train Decision Tree, Random Forest, KNN, and Gradient Boosting models, performing 5-fold cross-validation.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Gradient Boosting (XGB)": GradientBoostingClassifier(random_state=42)
}

best_acc = 0.0
best_model_name = ""
best_model = None

print("Model Evaluation Details:")
print("-" * 50)
for name, model in models.items():
    # 5-fold cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    
    print(f"{name}:")
    print(f"  CV Mean Accuracy: {np.mean(cv_scores):.4f}")
    print(f"  Test Accuracy:    {acc:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_model_name = name
        best_model = model
print("-" * 50)
print(f"Best Model: {best_model_name} with Accuracy {best_acc:.4f}")

### Step 8: Detailed Evaluation of Best Model & Export

In [ ]:
preds = best_model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))
print("\nClassification Report:")
print(classification_report(y_test, preds))

# Export the best model as rdf.pkl
with open('../rdf.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Saved best model as rdf.pkl")